# DeepDiscover: boundary exchanges and delayed routing

This example uses a water source, an externally prescribed pumping sink and a PET-limited sink. It also shows fixed, learned and dynamic routing. One physical process has exactly one destination in each configuration.

In [ ]:
import numpy as np
import tensorflow as tf
import keras
from deepdiscover import (
    DeepDiscover, CRCNet, DiscoverUnit, RoutingKernel,
    MultiDUModel, TrainableRoutingMultiDUModel,
    DynamicRoutingMultiDUModel, PrescribedSource,
    PrescribedSink, DemandLimitedSink,
)

n = 60
rng = np.random.default_rng(12)
rain = rng.gamma(1.3, 2.0, n).astype('float32')
pet = np.full(n, 0.8, dtype='float32')
lateral_recharge = 0.1 * rain
pumping = np.zeros(n, 'float32')
pumping[20:40] = 0.4
water = np.column_stack([rain, np.zeros(n, 'float32')])[None]
drivers = np.column_stack([rain, pet, lateral_recharge, pumping])[None]

In [ ]:
def build(model_type=MultiDUModel):
    loss_net = CRCNet([1, 1], [5, 100], output='sigmoid')
    first = DiscoverUnit(
        [CRCNet([1, -1, 1], [5, 3, 100])],
        context_indices=[(0, 1)],
        exchanges=[
            PrescribedSource(driver_index=2),
            DemandLimitedSink(loss_net, demand_index=1, context_indices=(0,)),
            PrescribedSink(driver_index=3),
        ],
    )
    second = DiscoverUnit(
        [CRCNet([1, -1, 1, 1], [5, 3, 100, 5])],
        context_indices=[(0, 1)], upstream_count=1,
    )
    physical = model_type(
        discover_units=[first, second], destinations=[[1], ['outlet']],
        routing=[[RoutingKernel([0.5, 0.5])], [RoutingKernel([0.4, 0.3, 0.3])]],
    )
    return DeepDiscover(physical)

for cls in (MultiDUModel, TrainableRoutingMultiDUModel, DynamicRoutingMultiDUModel):
    dd = build(cls)
    output = dd.simulate(water, drivers)
    error = np.max(np.abs(output['mass_residual'].numpy()))
    print(cls.__name__, 'maximum mass residual', error,
          'route weight shape', output['routing_weights'].shape)

## A custom source

User-defined sources and sinks are Keras layers. They declare which side of the water balance they represent. The output must be non-negative.

In [ ]:
class ImportedWater(keras.layers.Layer):
    kind = 'source'
    def call(self, drivers):
        return tf.nn.relu(drivers[:, 2:3])

class Extraction(keras.layers.Layer):
    kind = 'sink'
    def call(self, drivers, storage, available):
        return tf.minimum(tf.nn.relu(drivers[:, 3:4]), available)

custom_du = DiscoverUnit(
    [CRCNet([1, 1], [5, 100])],
    context_indices=[(0,)],
    exchanges=[ImportedWater(), Extraction()],
)
custom = DeepDiscover(MultiDUModel([custom_du], [['outlet']]))
custom_output = custom.simulate(water[:, :, :1], drivers)
assert np.max(np.abs(custom_output['mass_residual'].numpy())) < 1e-4
print('Imported source volume:', float(tf.reduce_sum(custom_output['source_total']).numpy()))
print('Extracted sink volume:', float(tf.reduce_sum(custom_output['sink_total']).numpy()))
print('Maximum numerical balance residual:', float(tf.reduce_max(tf.abs(custom_output['mass_residual'])).numpy()))

## Routing and storage continuation

If a run is interrupted at day 30, the next segment must begin with the previous water storage and the water already travelling along routes.

In [ ]:
dd = build(MultiDUModel)
full = dd.simulate(water, drivers)
first = dd.simulate(water[:, :30], drivers[:, :30])
second = dd.simulate(water[:, 30:], drivers[:, 30:],
                     initial_storage=first['final_storage'],
                     initial_memory=first['final_memory'],
                     initial_transit=first['final_transit_state'])
np.testing.assert_allclose(second['outlet'], full['outlet'][:, 30:], atol=1e-5)
print('Continuous and resumed simulations agree')